# Lab 2 — Student workbook: predicting with ECG, from features to foundation models

## 0. Setup

**Runtime.** *Runtime > Change runtime type > T4 GPU*; about ten minutes on the free T4.

In [ ]:
!pip install -q numpy matplotlib "neurokit2>=0.2.13" scikit-learn torch tqdm huggingface_hub pandas

### import relevant packages

In [ ]:
# Import Python basic libraries
import numpy as np
import matplotlib.pyplot as plt
import pickle as pk
import neurokit2 as nk
import tqdm

# Plot configurations
%config InlineBackend.figure_formats = {'png', 'retina'}
plt.rcParams['axes.labelsize'] = 16  # fontsize for figure labels
plt.rcParams['axes.titlesize'] = 18  # fontsize for figure titles


### get the lab's code and data
The cell below fetches the lab's code (`kit/code/`) and data (`data/`) the first time it runs, and asks for the data password, which your tutor gives out in the lab. Every input is one lead, 300 Hz, microvolts. The test sets' labels are not in these files.

In [ ]:
KIT_REPO = 'https://github.com/junjiechen-chris/CDT-ECG-Lab2'                    # the lab's code
DATA_ZIP_URL = 'https://huggingface.co/datasets/junjiechen-chris/CDT-HDS-ECG-Lab-2-data/resolve/main/ECG_lab2_data.zip'

import os, sys
KIT_DIR = os.environ.get('LAB2_KIT', 'kit/code')
if not os.path.exists(os.path.join(KIT_DIR, 'frontier_kit.py')):
    !git clone -q $KIT_REPO kit
sys.path.insert(0, KIT_DIR)

FILES = ['ECG.pkl', 'ECG_feature.pkl', 'alivecor_hidden.pkl', 'ptbxl_af.pkl', 'sex.pkl', 'apnea.pkl']
os.makedirs('data', exist_ok=True)
if any(not os.path.exists(f'data/{f}') for f in FILES):      # a password-protected zip; the password is given in the lab
    import getpass, zipfile
    from urllib.request import urlretrieve
    if not os.path.exists('data/ECG_lab2_data.zip'):
        print('downloading the data zip (431 MB) ...'); urlretrieve(DATA_ZIP_URL, 'data/ECG_lab2_data.zip')
    password = getpass.getpass('Password for the data zip: ')
    with zipfile.ZipFile('data/ECG_lab2_data.zip') as z:
        z.extractall('data', pwd=password.encode())

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using', torch.cuda.get_device_name(0) if device == 'cuda' else 'the CPU (slower; Runtime > Change runtime type > T4 GPU)')
import lab2_checks as checks

## 1. Preparing the three tasks

We prepare the three tasks so that every method is trained and evaluated in the same way.

### The three tasks and the benchmark

Every system is trained once per task and checked on a validation split. The **test sets are hidden**: other people, and for sex and apnea another country or sleep lab; only the leaderboard (Section 4) scores them.

| Task | Train | Validation | Hidden test | Metric |
|---|---|---|---|---|
| **AF**: N, A, O, ~, per recording | 1,500 PhysioNet 2017 recordings | 500 | 6,528 other PhysioNet 2017 recordings | macro F1 |
| **Sex**: female or male, per 10 s recording | 3,258 PTB-XL ECGs | 1,123 | 1,000 Chapman–Shaoxing ECGs | AUROC |
| **Apnea**: yes or no, per minute | 5,676 Apnea-ECG minutes | 1,939 | 3,471 UCD St Vincent's minutes | AUROC |

The leaderboard ranks a system by the **geometric mean** of its three hidden-set scores.

In [ ]:
from frontier_kit import CLASSES, alivecor_split, write_submission, check_submission
from systems import Data, tasks, system_rf, clean, to_files

D = Data('data')          # the recordings above, the four packs, and the AF split
import frontier_kit; frontier_kit._cache.update(alivecor_hidden=D.hid, ptbxl_af=D.ptb, sex=D.sex, apnea=D.apn)   # load each pack once
T = tasks(D)              # per task: raw signals, Lab 1 features, labels, train / validation indices, hidden targets
for name, t in T.items():
    print(f"{name:9s} train {len(t['tr']):5d}   validation {len(t['va']):5d}   hidden: "
          + ', '.join(f'{k} {len(v["raw"])}' for k, v in t['targets'].items()))

def split_by_person(person, val_frac=0.25, seed=0):
    """In: person, array (n,) of person ids, one per minute; val_frac, float; seed, int.
    Out: (train, val), int index arrays. A random val_frac of the people goes to validation with all their minutes."""
    people = np.unique(person)
    val_people = np.random.default_rng(seed).choice(people, int(round(val_frac * len(people))), replace=False)
    return np.where(~np.isin(person, val_people))[0], np.where(np.isin(person, val_people))[0]

def geomean(scores):
    """In: scores, 3 floats (AF, sex, apnea). Out: float, their geometric mean."""
    return float(np.exp(np.mean(np.log(scores))))

tr, va = split_by_person(D.apn['apnea_ecg_person'])
assert set(va) == set(D.apn['val'])
print(len(np.unique(D.apn['apnea_ecg_person'])), 'people;', len(np.unique(D.apn['apnea_ecg_person'][va])), 'in validation')

### System 1: your Lab 1 classifier on all three tasks

We fit a RandomForest on the 54 Lab 1 features of each task's training split; the features were computed in advance from the raw signals.

In [ ]:
val, preds = {}, {}       # validation scores and hidden-set predictions, filled system by system
preds['RF'], val['RF'] = system_rf(D)

## 2. End-to-end classification: ResNet 

Here a network reads the raw signal and learns its own filters from the labelled recordings; nobody designs a feature. One pipeline, four functions, serves all three tasks: `make_batch` (2.1), `build_model` (2.2), `predict_proba` (2.3, Task 2B), `train_resnet_classifier` (2.4). We call it on AF (2.4) and, unchanged, on sex and apnea (2.5).

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from model import resnet1d18
from systems import macro_f1, auroc

label_mapping = {c: i for i, c in enumerate(CLASSES)}   # N, A, O, ~ -> 0, 1, 2, 3, the order of every label array

### 2.1 Data preparation

We clean every recording with `nk.ecg_clean` as in Lab 1 (`clean`, supplied) and cut it to a fixed length with your `crop`.

### Task 2A Write `crop`

A network needs every input to have the same length; our recordings last 9 to 61 s. Training takes a random 10 s window, evaluation the first 10 s.

**Specification: `crop(x, length, train)`**
- **Given:** `x`, a 1-D array (one cleaned recording); `length`, the number of samples wanted (3,000 for 10 s at 300 Hz); `train`, True or False.
- **You may use:** `np.random.randint`, `np.pad`.
- **Return:** a 1-D array of exactly `length` samples. If `x` is longer than `length`: with `train=True`, the window `x[start:start + length]` with `start` drawn uniformly from 0 to `len(x) - length`; with `train=False`, the first `length` samples. If `x` is not longer, `x` followed by zeros up to `length`.
- **Checked by:** fixed signals (long, short, exactly `length`), and 30 random training windows.

In [ ]:
def crop(x, length, train):
    """In: x, 1-D float array (one cleaned ECG); length, int samples (3000 = 10 s); train, bool.
    Out: float array (length,). A random window if train, else the first `length` samples; a short x is zero-padded."""
    # YOUR CODE
    raise NotImplementedError

In [ ]:
crop = checks.check_crop(crop)

`make_batch` stacks the cropped windows of a batch into one tensor.

In [ ]:
def make_batch(signals, length, train):
    """In: signals, list of 1-D float arrays (cleaned ECG); length, int samples; train, bool.
    Out: float32 tensor (n, 1, length) on `device`. Each signal through crop(), then stacked."""
    windows = [crop(np.asarray(s, dtype=np.float32), length, train) for s in signals]
    windows = np.stack(windows)
    return torch.from_numpy(windows[:, None, :]).to(device)

clean_af = clean(T['alivecor']['raw'])        # the 2,000 AF recordings, cleaned once (about 20 s)
print(make_batch(clean_af[:4], 3000, train=True).shape)

### 2.2 The model

`build_model` returns the network (`resnet1d18`, one output per class), its optimiser (Adam) and its loss (cross-entropy), seeded so that a run repeats.

In [ ]:
def build_model(n_classes, lr=1e-3):
    """In: n_classes, int; lr, float.
    Out: (resnet1d18 with n_classes outputs, Adam optimizer, cross-entropy loss), seeded so a run repeats."""
    torch.manual_seed(0)
    np.random.seed(0)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    model = resnet1d18(num_classes=n_classes, input_channels=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    return model, optimizer, criterion

ecg_model, optimizer, criterion = build_model(n_classes=4)
print(sum(p.numel() for p in ecg_model.parameters()), 'coefficients')

### 2.3 Prediction

Signals in, class probabilities out: training needs it after every epoch, and Section 4 needs it for the hidden sets.

### Task 2B Write `predict_proba`

Write the forward pass: the first `length` samples of each signal (`make_batch` with `train=False`), through the network in batches, softmax to probabilities.

**Specification: `predict_proba(model, signals, length, batch_size=256)`**
- **Given:** `model`, a `resnet1d18` on `device`; `signals`, a list of cleaned 1-D arrays of any lengths; `length`, the number of samples per input; `batch_size`, how many signals to push through at once.
- **You may use:** `make_batch` (2.1), `model.eval()`, `torch.no_grad()`, `torch.softmax`, `np.concatenate`.
- **Return:** a 2-D array of shape (signals, classes): for each signal, the softmax of the model's scores on its first `length` samples (padded if shorter). No gradients are kept and the model is left in evaluation mode.
- **Checked by:** a small untrained network on five fixed random signals of different lengths, against a reference.

In [ ]:
def predict_proba(model, signals, length, batch_size=256):
    """In: model, resnet1d18; signals, list of 1-D float arrays (cleaned ECG); length, int samples; batch_size, int.
    Out: float array (n, n_classes), class probabilities. First `length` samples, eval mode, no gradients, softmax."""
    # YOUR CODE
    raise NotImplementedError

In [ ]:
predict_proba = checks.check_predict_proba(predict_proba)

### 2.4 Training

`train_resnet_classifier` runs the loop: each epoch trains on shuffled batches, then scores the validation split with your `predict_proba`; the best epoch's coefficients are kept.

In [ ]:
def train_resnet_classifier(signals, labels, train_idx, val_idx, n_classes,
                            length=None, epochs=20, batch_size=128):
    """In: signals, list of 1-D float arrays (cleaned ECG); labels, int array (n,); train_idx, val_idx, int arrays; n_classes, int;
    length, int samples (None: the longest); epochs, batch_size, int. Out: (trained resnet1d18, predict function, float best validation
    score: macro F1 if n_classes > 2 else AUROC). Each epoch trains on random crops, then scores validation; the best epoch is kept."""
    labels = np.asarray(labels)
    length = length or max(len(s) for s in signals)
    score = macro_f1 if n_classes > 2 else auroc
    model, optimizer, criterion = build_model(n_classes)
    predict = lambda sigs: predict_proba(model, sigs, length)  # noqa: E731

    best_score, best_state = -1.0, None
    rng = np.random.default_rng(0)
    for epoch in range(epochs):
        model.train()
        order = rng.permutation(train_idx)
        for i in range(0, len(order), batch_size):
            batch_idx = order[i:i + batch_size]
            optimizer.zero_grad()
            batch = make_batch([signals[j] for j in batch_idx], length, train=True)
            targets = torch.from_numpy(labels[batch_idx]).long().to(device)
            loss = criterion(model(batch), targets)
            loss.backward()
            optimizer.step()
        val_score = score(labels[val_idx], predict([signals[j] for j in val_idx]))
        print(f'epoch {epoch + 1}/{epochs}: validation {val_score:.3f}', flush=True)
        if val_score > best_score:
            best_score = val_score
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, predict, best_score

The training settings of the three tasks: the epochs, the batch size and the window length in samples (`None`: the whole recording). One call trains the AF network; about a minute on a T4. Then the validation report, and the predictions for the hidden sets.

In [ ]:
SETTINGS = {'alivecor': dict(epochs=20, batch_size=128, length=3000),   # 10 s windows
            'sex':      dict(epochs=3,  batch_size=128, length=3000),
            'apnea':    dict(epochs=2,  batch_size=32,  length=None)}   # whole 1-minute windows

from sklearn.metrics import classification_report

af = T['alivecor']
val['ResNet'], preds['ResNet'] = {}, {}
ecg_model, predict_af, val['ResNet']['alivecor'] = train_resnet_classifier(
    clean_af, af['y'], af['tr'], af['va'], af['n'], **SETTINGS['alivecor'])

val_pred = predict_af([clean_af[j] for j in af['va']]).argmax(1)
print(classification_report(af['y'][af['va']], val_pred, target_names=CLASSES))
print(f"ResNet, AF: validation macro F1 {val['ResNet']['alivecor']:.3f}   (RF: {val['RF']['alivecor']:.3f})")
for file, target in af['targets'].items():
    preds['ResNet'][file] = predict_af(clean(target['raw']))

One recording at a time: a list of one signal in, one row of probabilities out.

In [ ]:
p = predict_proba(ecg_model, [clean_af[0]], 3000)[0]      # A00001, label N
print(f'This recording is likely to be {CLASSES[int(p.argmax())]} with probability {p.max():.2f}')

### 2.5 Predicting sex and apnea with ECG and the ResNet

The same call on the other two tasks: two classes instead of four, and a smaller budget (three epochs for sex, two for apnea on whole 1-minute windows). About three minutes on a T4.

In [ ]:
for name in ['sex', 'apnea']:
    task, settings = T[name], SETTINGS[name]
    print(f'ResNet, {name}: {settings}')
    _, predict, val['ResNet'][name] = train_resnet_classifier(
        clean(task['raw']), task['y'], task['tr'], task['va'], task['n'], **settings)
    for file, target in task['targets'].items():
        preds['ResNet'][file] = predict(clean(target['raw']))
print({k: round(v, 3) for k, v in val['ResNet'].items()})

## 3. Foundation model classification: ECGFounder

A **foundation model** learned its features beforehand, from a far larger dataset. **ECGFounder** (Li et al., *NEJM AI* 2025; MIT licence) was pretrained on 10.8 million hospital ECGs to recognise **150 diagnostic statements**. For 10 s of signal it returns **1,024 numbers, an embedding**. We keep it **frozen** and train a linear classifier on the embedding. The cell below downloads it (370 MB, once) and embeds the 2,000 recordings.

In [ ]:
from ecgfounder import load_model, embed

fm_model = load_model(device)
E_af = embed(fm_model, D.raw_af)[0]     # (2000, 1024) embeddings
print(E_af.shape)

### 3.1 A linear classifier on the frozen embeddings, for the three tasks

### Task 3A Write `pool_windows`

ECGFounder reads 10 s at a time, so n one-minute recordings give E of shape (6n, 1024), six window embeddings per minute. Pool them by minute: the mean and the standard deviation over the six windows, side by side, give one (2048,) vector per minute, so (n, 2048) in all.

**Specification: `pool_windows(E, windows)`**
- **Given:** `E`, a 2-D array of `n * windows` rows (window embeddings) × `d` columns, where rows `i*windows` to `(i+1)*windows - 1` belong to item `i`; `windows`, the number of windows per item.
- **You may use:** `np.reshape`, `np.mean`, `np.std`, `np.concatenate`.
- **Return:** a 2-D array of `n` rows × `2d` columns: for each item, the mean of its windows (the first `d` columns), then their standard deviation (`ddof=0`, the last `d` columns).
- **Checked by:** a reference on a fixed random array.

In [ ]:
def pool_windows(E, windows):
    """In: E, float array (n * windows, d), item i in rows i*windows to (i+1)*windows - 1; windows, int.
    Out: float array (n, 2d): per item, the mean over its windows, then their standard deviation (ddof=0)."""
    # YOUR CODE
    raise NotImplementedError

In [ ]:
pool_windows = checks.check_pool_windows(pool_windows)

### Task 3B Write `linear_classifier`

We train a linear classifier using ECGFounder's frozen embedding. 

**Specification: `linear_classifier(E_train, y_train, E_eval)`**
- **Given:** `E_train`, a 2-D array of embeddings (one row per training item); `y_train`, their integer labels; `E_eval`, embeddings to predict.
- **You may use:** `StandardScaler` and `LogisticRegression` from scikit-learn.
- **Return:** a 2-D array of class probabilities, one row per row of `E_eval` (`predict_proba`), from a `LogisticRegression(C=0.01, max_iter=3000)` fitted on `E_train` after standardising each column with a scaler fitted on `E_train` only.
- **Checked by:** a reference on fixed random data; then its validation score on the three tasks.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def linear_classifier(E_train, y_train, E_eval):
    """In: E_train, float array (n_train, d); y_train, int array (n_train,); E_eval, float array (n_eval, d).
    Out: float array (n_eval, n_classes), class probabilities. Scaler fitted on E_train, LogisticRegression(C=0.01, max_iter=3000)."""
    # YOUR CODE
    raise NotImplementedError

In [ ]:
linear_classifier = checks.check_linear_classifier(linear_classifier)

In [ ]:
def embed_task(raw, windows=1):
    """In: raw, list of 1-D float arrays (raw ECG); windows, int pieces per recording.
    Out: float array (n, 1024), or (n, 2048) if windows > 1: each piece embedded, then pool_windows."""
    if windows == 1:
        return embed(fm_model, raw)[0]
    pieces = [p for s in raw for p in np.split(np.asarray(s, float), windows)]
    return pool_windows(embed(fm_model, pieces)[0], windows)

score = {'alivecor': macro_f1, 'sex': auroc, 'apnea': auroc}   # the task scores, as in Section 2
val['ECGFounder'], preds['ECGFounder'] = {}, {}
for name, windows in [('alivecor', 1), ('sex', 1), ('apnea', 6)]:
    t = T[name]
    E = E_af if name == 'alivecor' else embed_task(t['raw'], windows)
    val['ECGFounder'][name] = score[name](t['y'][t['va']], linear_classifier(E[t['tr']], t['y'][t['tr']], E[t['va']]))
    print(f"ECGFounder + linear classifier, {name}: validation {val['ECGFounder'][name]:.3f}")
    for file, target in t['targets'].items():
        preds['ECGFounder'][file] = linear_classifier(E[t['tr']], t['y'][t['tr']], embed_task(target['raw'], windows))

## 4. Beating the baselines

Three systems, three tasks: nine validation points below. The same nine points on the hidden sets are the baselines.

In [ ]:
import pandas as pd

table = pd.DataFrame(val).T[['alivecor', 'sex', 'apnea']].rename(columns={'alivecor': 'AF (macro F1)', 'sex': 'Sex (AUROC)', 'apnea': 'Apnea (AUROC)'})
table['geomean'] = [geomean(row) for row in table.values]
table.round(3)

The leaderboard scores the **hidden** sets. We pick an idea, implement it, check it on the validation split, and keep it if it helps. Some examples:
- **model ensembling**: average the probabilities of two systems (the cell below does that for RF and ECGFounder);
- **feature engineering**: give a classifier the 54 features *and* the embedding together;
- **model selection**: another classifier on the embedding, or other settings (`C`, number of trees);
- **hyperparameter tuning**: more epochs, another learning rate, another crop length for the ResNet (`SETTINGS` in 2.4);
- **fine-tuning the foundation model**: train ECGFounder end to end on our labels (`ecgfounder.finetune`, a few minutes per task on a T4).

### Task 4 Beat the baselines
Put your name below. `write_submission` writes your predictions on the three test sets as `alivecor.csv`, `sex.csv` and `apnea.csv` and zips them with your `notes`; `check_submission` checks the zip. Upload it on the leaderboard page; every upload is kept, so submit often.

In [ ]:
TEAM = '<YOUR NAME>'   # appears on the leaderboard
NOTE="""
<YOUR NOTE>
"""

# Your first entry: the average of the RF and ECGFounder probabilities, per task
mine = {k: (preds['RF'][k] + preds['ECGFounder'][k]) / 2 for k in preds['ECGFounder']}
z = write_submission(f'{TEAM}-submission', author=TEAM, notes=NOTE, **to_files(mine))
check_submission(z)

## Submit your work
Submit two things: this notebook with your four functions implemented (restart the kernel and run all cells first), and your test-set predictions as the three files of Task 4, with a short note of what you tried.